In [1]:
%load_ext autoreload
%autoreload 2
%config Completer.use_jedi = False

import os
from getpass import getpass
import pandas as pd
import json
import re
from tqdm import tqdm
from sklearn.metrics import classification_report
from json.decoder import JSONDecodeError

import nest_asyncio
from dotenv import load_dotenv
from openai import OpenAI

pd.set_option('max_colwidth', 400)

nest_asyncio.apply()

load_dotenv('.env', verbose=True)

True

Для работы с llm через API будем использовать **OpenRouter**

Создайте API ключ на https://openrouter.ai/keys

**Способ 1 (безопасный):** Создайте файл .env и добавьте:
- `OPENROUTER_API_KEY=ваш_ключ`

**Способ 2:** Вставьте ключ напрямую в ноутбук в следующей ячейке:
- `openrouter_api_key = "<ваш ключ>"`

In [2]:
# Способ 1: Загрузка из переменных окружения 
openrouter_api_key = os.environ.get('OPENROUTER_API_KEY')

# Способ 2: Вставить ключ напрямую 
# openrouter_api_key = "<ваш ключ>"

In [3]:
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=openrouter_api_key,
)
model_name = "qwen/qwen-2.5-7b-instruct"  

In [4]:
def clean_json_response(response: str) -> str:
    """
    Убираем в ответе llm markdown (```json и ```)
    """
    response = response.strip()
    if response.startswith('```json'):
        response = response[7:] 
    elif response.startswith('```'):
        response = response[3:] 
    
    if response.endswith('```'):
        response = response[:-3]
    
    return response.strip()

In [70]:
def get_completion(prompt: str, system_prompt=""):
    messages = []
    if system_prompt:
        messages.append({
            "role": "system",
            "content": system_prompt
        })
    messages.append({
        "role": "user",
        "content": prompt
    })
    
    response = client.chat.completions.create(
        model=model_name,
        messages=messages,
        max_tokens=500,      # Вернули место для рассуждений
        temperature=0.0,     # Оставили 0.0 для точности
        top_p=0.9,
    )
    return response.choices[0].message.content

In [71]:
get_completion("Привет!")

'Привет! Как я могу вам помочь сегодня?'

## Предсказание категории

In [73]:
categories = [
    "Ролики и скейтбординг",
    "Настольные игры",
    "Теннис бадминтон пинг-понг",
    "Пейнтбол и страйкбол",
    "Единоборства",
    "Бильярд и боулинг",
    "Фитнес и тренажёры",
    "Туризм",
    "Игры с мячом",
    "Зимние виды спорта",
    "Дайвинг и водный спорт",
    "Спортивное питание",
]

In [74]:
import re


In [75]:
categories_list_str = "\n".join([f"- {cat}" for cat in categories])

system_prompt = f"""Ты — аналитик маркетплейса. Твоя цель — классифицировать спортивный товар в СТРОГО валидный JSON.

СПИСОК ДОПУСТИМЫХ КАТЕГОРИЙ:
{categories_list_str}

ПРАВИЛА:
1. Твой ответ должен быть ТОЛЬКО в формате JSON, начинаться с {{ и заканчиваться на }}. Никакого Markdown (```json).
2. В JSON строго два поля: "reasoning" (краткое рассуждение) и "category" (категория СИМВОЛ В СИМВОЛ из списка).
3. СПЕЦ-ПРАВИЛО: Любое военное/тактическое снаряжение (кобура, разгрузка, подсумок, пульки, Ратник, ССО, магазины) относится к "Пейнтбол и страйкбол".
4. СПЕЦ-ПРАВИЛО: БАДы, витамины, протеин, омега, креатин — это "Спортивное питание".

ПРИМЕРЫ:
Товар: <text>Футзалки lunar gato</text>
{{"reasoning": "Футзалки — это обувь для игры в футбол", "category": "Игры с мячом"}}

Товар: <text>Маты татами</text>
{{"reasoning": "Татами используется для занятий боевыми искусствами", "category": "Единоборства"}}

Товар: <text>Подсумок для сбора магазинов Mag Net hsgi</text>
{{"reasoning": "Подсумки и магазины — это военная экипировка для тактических игр", "category": "Пейнтбол и страйкбол"}}

Товар: <text>Карбоновый шафт 12 мм (пул)</text>
{{"reasoning": "Пул — это вид бильярда, шафт — часть кия", "category": "Бильярд и боулинг"}}

Товар: <text>Бамбинтон СССР</text>
{{"reasoning": "Опечатка в слове бадминтон", "category": "Теннис бадминтон пинг-понг"}}

Товар: <text>Эмалированные кружки</text>
{{"reasoning": "Походная посуда", "category": "Туризм"}}
"""

In [76]:
# test_df и val_df лежат на степике
test_df = pd.read_csv('test_df_llm_generation_1.csv')[['title', 'item_id']]
val_df = pd.read_csv('val_df_llm_generation.csv')
test_df.sample(10)

,title,item_id
39,Лента для хоккейной клюшки,39
66,Пояс тренера Reyvel,66
106,Гамаши тактические (Олива/Цифра/Мох/Мультикам),106
37,Bauer Mach хоккеные краги / перчатки,37
61,SKI mask Rossignol,61
25,Шорты с защитой Nike Pro Hyper Strong,25
88,Мощный предтреник Hazard Core Pre Ashes,88
2,Маты татами,2
119,Драйн гринвей,119
5,Омега,5


In [77]:
def process_dataframe_with_cache(test_df, system_prompt, cache_file='answer_cache.json'):
    key_to_answer = {}
    if os.path.exists(cache_file):
        with open(cache_file, 'r', encoding='utf-8') as f:
            key_to_answer = json.load(f)
    answers = []
    
    for i, row in tqdm(test_df.iterrows(), total=len(test_df)):
        title = row['title']
        key = f"v4_{title}" # НОВЫЙ КЛЮЧ! Начнем с чистого листа автоматически
        
        if key in key_to_answer:
            answer = key_to_answer[key]
        else:
            prompt = f"Товар: <text>{title}</text>"
            answer = get_completion(prompt=prompt, system_prompt=system_prompt)
            key_to_answer[key] = answer
            with open(cache_file, 'w', encoding='utf-8') as f:
                json.dump(key_to_answer, f, ensure_ascii=False, indent=2)
        answers.append(answer)
    
    return answers

In [79]:
answers = process_dataframe_with_cache(test_df, system_prompt)

100%|██████████| 121/121 [06:45<00:00,  3.35s/it]


In [ ]:
predicted_categories = []
valid_categories = set(categories)

for answer in answers:
    try:
        # Ищем всё между { и } включительно
        match = re.search(r'\{.*\}', answer, re.DOTALL)
        if match:
            clean_str = match.group(0)
        else:
            clean_str = '{"category": "other"}'
            
        # strict=False спасает от ошибок спецсимволов
        parsed_answer = json.loads(clean_str, strict=False)
        predicted_category = parsed_answer.get('category', 'other')
        
        # Защита от галлюцинаций (выдуманных категорий)
        if predicted_category not in valid_categories:
            predicted_category = 'other'
            
    except Exception as e:
        predicted_category = 'other'
        
    predicted_categories.append(predicted_category)

# Смотрим, сколько не распознано
other_count = predicted_categories.count('other')
print(f"Сброшено в other: {other_count} из {len(predicted_categories)}")

In [ ]:
predicted_categories = []
for answer in answers:
    try:
        # Очищаем ответ markdown
        cleaned_answer = clean_json_response(answer)
        parsed_answer = json.loads(cleaned_answer)
        predicted_category = parsed_answer.get('category', 'other')
        if 'category' not in parsed_answer:
            print(f"Отсутствует ключ 'category'! Ответ модели: {cleaned_answer}")
    except JSONDecodeError as e:
        print(f"Ошибка парсинга JSON: {e}")
        print(f"Ответ: {answer[:100]}...")
        predicted_category = 'other'
    predicted_categories.append(predicted_category)

In [69]:
test_df_gpt = test_df.copy()
test_df_gpt['category'] = predicted_categories
test_df_gpt[['item_id', 'category']].to_csv('test_df_to_upload.csv', index=False)
test_df_gpt.to_csv('delete_me.csv', index=False)

### Ваше решение